# 声の似かた 比較実験（Colab 用）

**目的**：翻訳した声が、本人の声にどれだけ似ているか（現状値）を測り、改善策を **1つずつ** 試して効果を比べます。

結果は Excel（`voicelab_result.xlsx`）にまとまります。まず「現状値」シートの数字を見てください。

| 条件 | 何をするか |
|---|---|
| C0 `line_ref_16k` | 今の pyVideoTrans と同じ：短い（2〜4秒）・16kHz の手本。**これが現状値** |
| C1 `profile_ref` | 登録した良い手本（10〜15秒・元の音質・書き起こし付き）に替える |
| C2 `stretch` | C1 の音声を 1.3倍・2倍の速さにして、早回しで声がどれだけ変わるか見る |
| C3 `emotion` | VoxCPM2 で、感情の指示を入れると似かたがどれだけ下がるか見る |

**所要時間の目安（T4 GPU）**
- お試し（`quick`）：準備 15〜30分（初回だけ）＋ 生成 10〜20分 ＋ 測定 5〜10分
- 本番（`full`、1人）：生成 1〜2時間 ＋ 測定 30分〜1時間
- Colab の無料枠は途中で切れることがあります。結果は Google ドライブに保存され、同じ手順で **続きから再開** できます。

## ⚠️ 必ず読んでください：Colab に上げてよい音声

Colab はクラウド（Google のサーバー）です。次の3つだけにしてください。

1. **自分自身の声**
2. **本人の同意を得た模擬会議の音声**
3. **公開データ**（LibriSpeech など）

**会社の会議音声・社外秘の音声は、絶対にアップロードしないでください。**
また、声の複製は **本人の同意** が必要です。他人の声を無断で使わないでください。

In [ ]:
#@title 0. 上の注意を読みました（チェックを入れて実行）
I_AGREE = False  #@param {type:"boolean"}
assert I_AGREE, "上の注意を読んで、チェックを入れてからもう一度実行してください（自分の声／同意済みの模擬会議／公開データだけを使います）"
print("OK：自分の声・同意済みの模擬会議・公開データだけを使います。")

## 1. GPU を確認する

In [ ]:
#@title 1. GPU の確認
import shutil, subprocess

HAS_GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
if HAS_GPU:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout
    print("GPU が使えます：", out.strip())
else:
    print("GPU が見つかりません。メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、最初からやり直してください。")
    print("（GPU がなくても、手順 8 の「ダミーでの確認」までは進められます）")

## 2. Google ドライブをつなぐ

Colab は一定時間でリセットされます。結果をなくさないために、**出力は Google ドライブ**（`マイドライブ/voicelab/`）に書きます。

In [ ]:
#@title 2. Google ドライブをつなぐ
from google.colab import drive

drive.mount("/content/drive")

## 3. 設定

ここの値だけ変えれば動きます。最初は **`MODE = "quick"`（お試し）** と **`SPEAKER_SOURCE = "public_test"`（公開データ）** のまま全部実行するのがおすすめです。

- `MODE`：`quick` は繰り返し1回・文は言語ごとに1つ。`full` は繰り返し2回・全部の文。
- `SPEAKER_SOURCE`：
  - `public_test` … 公開データ（LibriSpeech 1人）で最初の動作確認
  - `drive_profile` … voiceprofile ツールで作ったフォルダ（`profile.json` がある）をドライブに置いた場合
  - `drive_folder` … 自分の声の wav と書き起こし（`transcripts.csv`）をドライブのフォルダに置いた場合
  - `upload` … この画面から wav（と `transcripts.csv`）をアップロードする場合
- エンジンは1つずつ別の環境に入れます（ライブラリの版が互いに衝突するため）。使わないものは `False` にしてください。
  `USE_COSYVOICE3` はインストールが一番壊れやすいので、最初は `False` のままで構いません。

In [ ]:
#@title 3. 設定
EXP_NAME = "exp01"  #@param {type:"string"}
SPEAKER_NAME = "A"  #@param {type:"string"}
SPEAKER_SOURCE = "public_test"  #@param ["public_test", "drive_profile", "drive_folder", "upload"]
MODE = "quick"  #@param ["quick", "full"]
REF_LANG = ""  #@param {type:"string"}
#   REF_LANG：手本の言語を決め打ちしたいとき（例 ja）。空なら自動

USE_VOXCPM2 = True  #@param {type:"boolean"}
USE_QWEN3TTS = True  #@param {type:"boolean"}
USE_CHATTERBOX = True  #@param {type:"boolean"}
USE_COSYVOICE3 = False  #@param {type:"boolean"}

EMBEDDER = "ecapa"  #@param ["ecapa", "resemblyzer", "all", "none"]
DRIVE_ROOT = "/content/drive/MyDrive/voicelab"  #@param {type:"string"}
CACHE_MODELS_ON_DRIVE = False  #@param {type:"boolean"}
#   True にすると、ダウンロードしたモデルをドライブに保存して次回の時間を節約（ドライブの空きが数GB〜十数GB必要）

import os

REPO = "/content/pyvideotrans"
SRC = f"{REPO}/meeting/src"
OUT_ROOT = f"{DRIVE_ROOT}/{EXP_NAME}"          # 設定・文・結果の置き場所（ドライブ）
RESULT_DIR = f"{OUT_ROOT}/result"
CFG_PATH = f"{OUT_ROOT}/exp.json"
ENROLL_DIR = f"{DRIVE_ROOT}/enroll"
HF_HOME = f"{DRIVE_ROOT}/hf_cache" if CACHE_MODELS_ON_DRIVE else "/content/hf_cache"
os.makedirs(OUT_ROOT, exist_ok=True)

ENGINES = []
if USE_VOXCPM2: ENGINES.append("voxcpm2")
if USE_QWEN3TTS: ENGINES.append("qwen3tts")
if USE_COSYVOICE3: ENGINES += ["cosyvoice3:zero_shot", "cosyvoice3:cross_lingual"]
if USE_CHATTERBOX: ENGINES.append("chatterbox")
assert ENGINES, "エンジンを1つ以上 True にしてください"
EMOTION_ENGINES = ["voxcpm2"] if USE_VOXCPM2 else []
REPEATS, MAX_TEXTS = (1, 1) if MODE == "quick" else (2, None)
print("エンジン：", ENGINES)
print("出力先（ドライブ）：", OUT_ROOT)

## 4. プログラムを取ってくる

GitHub の `mass-spring/pyvideotrans`（ブランチ `meeting-translator`）をクローンします。
**非公開リポジトリの場合** は、GitHub の Personal access token（読み取り権限）を聞かれるので貼り付けてください（画面には表示されません）。

In [ ]:
#@title 4. リポジトリをクローンする
import os, subprocess, getpass

SECRETS = []   # 画面（と保存されるノートブック）に出したくない文字列。token など

def sh(cmd, env=None, cwd=None):
    """コマンドを実行して、出力をそのまま表示する。終了コードを返す。"""
    e = os.environ.copy()
    e.update(env or {})
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=e, cwd=cwd, bufsize=1)
    for line in p.stdout:
        for sec in SECRETS:
            line = line.replace(sec, "***")
        print(line, end="")
    return p.wait()

URL = "https://github.com/mass-spring/pyvideotrans"
if os.path.isdir(f"{REPO}/.git"):
    sh(["git", "-C", REPO, "fetch", "origin", "meeting-translator"])
    sh(["git", "-C", REPO, "checkout", "meeting-translator"])
    sh(["git", "-C", REPO, "pull", "--ff-only", "origin", "meeting-translator"])
else:
    rc = sh(["git", "clone", "--depth", "1", "--branch", "meeting-translator", URL, REPO])
    if rc != 0:
        token = getpass.getpass("非公開リポジトリのようです。GitHub の token を貼り付けてください（空なら中止）：")
        assert token, "クローンできませんでした"
        SECRETS.append(token)
        assert sh(["git", "clone", "--depth", "1", "--branch", "meeting-translator",
                   f"https://{token}@github.com/mass-spring/pyvideotrans", REPO]) == 0, "クローンに失敗しました"
assert os.path.isdir(SRC), f"{SRC} が見つかりません"
print("OK：", SRC)

## 5. 実行環境を作る

エンジンごとに別の「仮想環境」を作ります（`uv` を使います）。**初回は数分〜10分ほどかかります**。2回目以降は速いです。

- 5-0：共通の道具と、測定用の環境（`score`）。**必ず実行**
- 5-1〜5-4：使うエンジンだけ実行（上の `USE_...` が `True` のもの）。うまく入らなくても、他のエンジンは続けられます。

入れるバージョンは、PyPI に存在することを確認したものに固定しています。

In [ ]:
#@title 5-0. 共通の道具と測定用の環境（score）
sh(["pip", "-q", "install", "uv==0.11.32"])
sh("apt-get -qq update && apt-get -qq install -y rubberband-cli sox libsox-dev ffmpeg")  # 早回しと音声処理の部品

ENV_DIR = "/content/envs"
os.makedirs(ENV_DIR, exist_ok=True)
os.makedirs(HF_HOME, exist_ok=True)

def env_python(name):
    return f"{ENV_DIR}/{name}/bin/python"

def make_env(name, python, packages, pre=()):
    """仮想環境 name を作り、packages を入れる。pre は先に入れるもの（別のインデックス用）。成功すれば True。"""
    py = env_python(name)
    if not os.path.exists(py):
        if sh(["uv", "venv", "--python", python, f"{ENV_DIR}/{name}"]) != 0:
            print(f"× 仮想環境 {name} を作れませんでした"); return False
    for args in pre:
        if sh(["uv", "pip", "install", "--python", py, *args]) != 0:
            print(f"× {name}：{args} を入れられませんでした"); return False
    if sh(["uv", "pip", "install", "--python", py, *packages]) != 0:
        print(f"× {name}：インストールに失敗しました。上のメッセージを確認してください"); return False
    print(f"○ 環境 {name} ができました"); return True

def check_import(name, module):
    rc = sh([env_python(name), "-c", f"import {module}; print('import {module}: OK')"])
    return rc == 0

# 測定用（CPU 版の torch でよい）。voicecheck（ecapa）と voicelab の「測定」「ダミー確認」がここで動く
SCORE_OK = make_env(
    "score", "3.12",
    ["numpy", "scipy", "soundfile", "librosa", "openpyxl", "pandas", "praat-parselmouth", "speechbrain==1.0.3"],
    pre=[["--index-url", "https://download.pytorch.org/whl/cpu", "torch==2.6.0", "torchaudio==2.6.0"]],
)
# 声の高さを正確に測る pyworld（入らなくても動く。その場合は librosa.pyin で代用するので測定が遅くなる）
if SCORE_OK and sh(["uv", "pip", "install", "--python", env_python("score"), "setuptools<81", "pyworld"]) != 0:
    print("pyworld が入りませんでした。声の高さは librosa.pyin で測ります（遅いので、quick での利用がおすすめ）")

In [ ]:
#@title 5-1. VoxCPM2（Apache-2.0）を入れる
if USE_VOXCPM2:
    if make_env("voxcpm2", "3.12", ["torch==2.6.0", "torchaudio==2.6.0", "voxcpm==2.0.3", "numpy", "soundfile", "scipy", "librosa"]):
        check_import("voxcpm2", "voxcpm")
else:
    print("VoxCPM2 は使いません（スキップ）")

In [ ]:
#@title 5-2. Qwen3-TTS（Apache-2.0）を入れる
if USE_QWEN3TTS:
    if make_env("qwen3tts", "3.12", ["torch==2.6.0", "torchaudio==2.6.0", "qwen-tts==0.1.1", "numpy", "soundfile", "scipy", "librosa"]):
        check_import("qwen3tts", "qwen_tts")
else:
    print("Qwen3-TTS は使いません（スキップ）")

In [ ]:
#@title 5-3. Chatterbox Multilingual（MIT）を入れる
if USE_CHATTERBOX:
    # chatterbox-tts 0.1.7 は torch==2.6.0 / transformers==5.2.0 / numpy<2 を固定で要求する（だから別環境）
    # 透かし部品 resemble-perth が pkg_resources（setuptools<81 に含まれる）を使うので一緒に入れる。無いと読み込みで TypeError になる
    if make_env("chatterbox", "3.11", ["chatterbox-tts==0.1.7", "soundfile", "scipy", "setuptools<81"]):
        check_import("chatterbox", "chatterbox.mtl_tts")
else:
    print("Chatterbox は使いません（スキップ）")

In [ ]:
#@title 5-4. Fun-CosyVoice3（Apache-2.0）を入れる【一番壊れやすい】
# pip では入らないので、公式リポジトリをクローンして requirements.txt を入れる（公式は Python 3.10）。
# 推論に要らない重い部品（deepspeed・tensorrt・gradio・fastapi・grpc など）は除く。Colab での動作は未確認。
COSY_REPO = "/content/CosyVoice"
if USE_COSYVOICE3:
    if not os.path.isdir(f"{COSY_REPO}/cosyvoice"):
        sh(["git", "clone", "--recursive", "https://github.com/FunAudioLLM/CosyVoice.git", COSY_REPO])
    drop = ("--", "#", "deepspeed", "tensorrt", "gradio", "fastapi", "grpcio", "tensorboard", "uvicorn", "pyworld")
    reqs = [l.strip() for l in open(f"{COSY_REPO}/requirements.txt", encoding="utf-8")
            if l.strip() and not l.strip().lower().startswith(drop)]
    if make_env("cosyvoice3", "3.10", reqs + ["soundfile", "scipy", "huggingface_hub", "pykakasi==2.3.0"]):
        check_import("cosyvoice3", "onnxruntime")
        print("モデル（約数GB）は、最初の生成のときに自動でダウンロードされます。")
else:
    print("CosyVoice3 は使いません（スキップ）")

## 6. 合成する文（targets.csv）

会議風の文を日本語・英語・フランス語・中国語で3つずつ、感情のある文（怒り・喜び・悲しみ）を各言語で1つずつ用意しています。
ドライブの `targets.csv` は **Excel やスプレッドシートで直しても構いません**（すでにあれば上書きしません）。
列：`id`（名前）、`lang`（言語）、`kind`（`meeting` か `emotional`）、`emotion`（感情の指示。英語）、`text`（読み上げる文）。

In [ ]:
#@title 6. 合成する文を用意する
import sys
sys.path.insert(0, SRC)          # voicelab を import できるようにする（pip install -e の代わり。Python のバージョン制限を避けるため）
os.environ["PYTHONPATH"] = SRC

import pandas as pd
from voicelab.texts import write_targets

TARGETS_CSV = f"{OUT_ROOT}/targets.csv"
if not os.path.exists(TARGETS_CSV):
    write_targets(TARGETS_CSV)
    print("既定の文を書きました：", TARGETS_CSV)
else:
    print("すでにある targets.csv を使います：", TARGETS_CSV)
pd.set_option("display.max_colwidth", 80)
display(pd.read_csv(TARGETS_CSV, encoding="utf-8-sig"))

## 7. 話者の声を用意する

`SPEAKER_SOURCE` で選んだ方法で、話者の登録音声を用意します。**該当する1つのセルだけ** 実行してください（自動で選ばれます）。

- 手本（登録音声）… 10〜15秒で、書き起こしがあるもの
- 評価用の実声 … **手本に使っていない** 本人の発話が 4 本以上（3〜6秒くらい）。これと比べて「似ているか」を測ります

In [ ]:
#@title 7. 話者の声を用意する
import glob, io, json, shutil
SPEAKER_SPEC = None

if SPEAKER_SOURCE == "public_test":
    # 公開データ（LibriSpeech 由来、CC-BY-4.0）：英語の 1 人。最初の動作確認用
    sh(["pip", "-q", "install", "pyarrow", "huggingface_hub", "soundfile"])
    sys.path.insert(0, f"{SRC}/scripts")
    import prepare_testdata as ptd
    ptd.dummy()                                            # 公開音声をダウンロード（話者 1272 の 73 発話）
    import pyarrow.parquet as pq, soundfile as sf, csv
    folder = f"{ENROLL_DIR}/public1272"
    os.makedirs(folder, exist_ok=True)
    rows = []
    for pf in sorted(glob.glob(f"{ptd.ROOT}/librispeech_asr_dummy/clean/*.parquet")):
        for r in pq.read_table(pf).to_pylist():
            data, sr = sf.read(io.BytesIO(r["audio"]["bytes"]), dtype="int16")
            name = f"{r['id']}.wav"
            sf.write(f"{folder}/{name}", data, sr, subtype="PCM_16")
            t = r["text"].strip().lower()
            rows.append({"file": name, "text": t[:1].upper() + t[1:] + ".", "lang": "en"})
    with open(f"{folder}/transcripts.csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file", "text", "lang"]); w.writeheader(); w.writerows(rows)
    SPEAKER_SPEC = {"name": "public1272", "folder": folder}
    print(f"公開データ {len(rows)} 発話を用意しました：{folder}")

elif SPEAKER_SOURCE == "drive_profile":
    folder = f"{DRIVE_ROOT}/profiles/{SPEAKER_NAME}"
    assert os.path.exists(f"{folder}/profile.json"), f"{folder}/profile.json がありません。voiceprofile で作ったフォルダを置いてください"
    SPEAKER_SPEC = {"name": SPEAKER_NAME, "profile": folder}
    print("プロフィールを使います：", folder)

elif SPEAKER_SOURCE == "drive_folder":
    folder = f"{ENROLL_DIR}/{SPEAKER_NAME}"
    assert glob.glob(f"{folder}/*.wav"), f"{folder} に wav がありません"
    SPEAKER_SPEC = {"name": SPEAKER_NAME, "folder": folder}
    print("フォルダを使います：", folder, "（transcripts.csv か、wav と同名の .txt に書き起こしを置く）")

elif SPEAKER_SOURCE == "upload":
    from google.colab import files
    print("自分の声の wav を選んでください。transcripts.csv（列：file,text,lang）も一緒に選べます。")
    up = files.upload()
    folder = f"{ENROLL_DIR}/{SPEAKER_NAME}"
    os.makedirs(folder, exist_ok=True)
    for fn, data in up.items():
        with open(f"{folder}/{fn}", "wb") as f:
            f.write(data)
    SPEAKER_SPEC = {"name": SPEAKER_NAME, "folder": folder}
    print(f"{len(up)} 個を {folder} に保存しました（ドライブにも残ります）")

assert SPEAKER_SPEC, "SPEAKER_SOURCE の値を確認してください"

## 8. 設定ファイルを書いて、何を作るか確認する（ここまでは GPU を使いません）

In [ ]:
#@title 8-1. 設定ファイル（exp.json）を書き、作る数を確認する
CFG = {
    "name": EXP_NAME,
    "speakers": [SPEAKER_SPEC],
    "targets": TARGETS_CSV,
    "target_langs": ["ja", "en", "fr", "zh"],
    "engines": ENGINES,
    "conditions": ["line_ref_16k", "profile_ref", "stretch", "emotion"],
    "emotion_engines": EMOTION_ENGINES,
    "repeats": REPEATS,
    "seed": 1234,
    "stretch_factors": [1.3, 2.0],
    "max_texts_per_lang": MAX_TEXTS,
    "ref_lang": REF_LANG,
    "embedder": EMBEDDER,
}
with open(CFG_PATH, "w", encoding="utf-8") as f:
    json.dump(CFG, f, ensure_ascii=False, indent=2)
print("設定を書きました：", CFG_PATH)

def vl(env, *args):
    """voicelab を、仮想環境 env の Python で実行する。"""
    return sh([env_python(env), "-u", "-m", "voicelab", *args],
              env={"PYTHONPATH": SRC, "HF_HOME": HF_HOME, "COSYVOICE_REPO": "/content/CosyVoice"})

vl("score", "plan", "--config", CFG_PATH, "--out", RESULT_DIR)

In [ ]:
#@title 8-2. ダミーで流れだけ確認する（モデル不要・CPU・1〜3分）【おすすめ】
# 本物のモデルは使わず、手本を加工したダミー音声で「生成 → 測定 → Excel」まで通します。
# ここで出る数字に意味はありません。設定や声のデータの間違いを、先に見つけるための確認です。
import copy
DRY_CFG = copy.deepcopy(CFG)
DRY_CFG.update({"engines": ENGINES[:1], "repeats": 1, "max_texts_per_lang": 1, "embedder": "none"})
DRY_PATH = "/content/dry_exp.json"
with open(DRY_PATH, "w", encoding="utf-8") as f:
    json.dump(DRY_CFG, f, ensure_ascii=False)
rc = vl("score", "run", "--config", DRY_PATH, "--out", "/content/dryrun", "--dry-run", "--embedder", "none")
print("ダミーでの確認：", "成功" if rc == 0 else "失敗（上のエラーを確認してください）")

## 9. 生成する（GPU）

選んだエンジンを **1つずつ** 別のプロセスで動かします。1つ終わるごとに GPU のメモリは空になります（T4 16GB で順番に動かすため）。

- 途中で止まっても、**同じセルをもう一度実行** すれば、できた分は飛ばして続きから作ります。
- 1つのエンジンが失敗しても、他のエンジンは続けます。失敗の理由は `result/manifest.csv` の `error` 列に残ります。
- 初回は各モデルのダウンロード（数GB）が入るため、最初の1件までに数分かかります。

In [ ]:
#@title 9. 音声を生成する（GPU）
GROUPS = [("voxcpm2", USE_VOXCPM2), ("qwen3tts", USE_QWEN3TTS), ("cosyvoice3", USE_COSYVOICE3), ("chatterbox", USE_CHATTERBOX)]
assert HAS_GPU, "GPU がありません。手順 1 を確認してください"
for env, use in GROUPS:
    if not use:
        continue
    if not os.path.exists(env_python(env)):
        print(f"--- {env}：環境がありません（手順 5 を実行してください）。飛ばします"); continue
    print(f"\n========== {env} ==========")
    vl(env, "generate", "--config", CFG_PATH, "--out", RESULT_DIR, "--engines", env)
    sh(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"])   # 終わったら空いているはず

man = pd.read_csv(f"{RESULT_DIR}/manifest.csv", encoding="utf-8-sig")
display(man.groupby(["engine", "status"]).size().unstack(fill_value=0))

## 10. 測定して Excel を作る（CPU）

生成した音声を、評価用の実声（手本に使っていないもの）と比べて、似ている度合い（SIM）を出します。
音声の数が多いと時間がかかります（目安：音声1秒あたり約0.5秒。`quick` なら数分）。

In [ ]:
#@title 10. 測定して Excel を作る
args = ["score", "--config", CFG_PATH, "--out", RESULT_DIR, "--embedder", EMBEDDER]
rc = vl("score", *args)
XLSX = f"{RESULT_DIR}/voicelab_result.xlsx"
print("\n結果：", XLSX if rc == 0 else "失敗（上のエラーを確認してください）")

## 11. 結果を見る

- **現状値**（`line_ref_16k` の行）… 今の方式でどれだけ似るか。`real` の行は本人の実声どうし（上限の目安）。
- **条件を変えたときの差** … プラスなら、その条件のほうが似ている（改善）。
- 差が「ばらつき」より小さいときは、偶然の揺れかもしれません。

In [ ]:
#@title 11. まとめの表を表示する
xl = pd.ExcelFile(XLSX)
for sheet in ["現状値", "条件比較"]:
    print(f"\n===== {sheet} =====")
    display(pd.read_excel(xl, sheet_name=sheet, header=None).dropna(how="all").fillna(""))

In [ ]:
#@title 12. Excel をダウンロードする
from google.colab import files
files.download(XLSX)
print("ドライブにも残っています：", XLSX)

## うまくいかないとき

| 症状 | 対処 |
|---|---|
| `GPU が見つかりません` | 「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」。変更後は最初からやり直す |
| 5-x のインストールが失敗する | そのエンジンだけ `USE_...` を `False` にして続ける（他のエンジンの結果は出ます）。失敗メッセージは `README` の「困ったとき」を参照 |
| 9 で `エンジンを読み込めません` | 5-x を実行したか確認。メモリ不足なら「ランタイム」→「セッションを再起動」して 9 から再実行 |
| 9 の途中で Colab が切れた | もう一度 3・4・5-0・6・7・8-1 を実行してから 9 を実行すると、続きから作ります（結果はドライブ上） |
| 10 の測定が遅い | `MODE = "quick"` にする。pyworld が入らなかった場合は特に遅い |
| 評価用の実声が足りないというエラー | 手本に使っていない本人の発話を 4 本以上用意する（README 参照） |
| `ecapa: モデルを取得できません` | 初回だけ HuggingFace からモデルをダウンロードします。ネットワークを確認して再実行 |